# XGBoost Hyperparameter Tuning — 3-Class ATSEG Classifier

This notebook tunes and evaluates an **XGBoost multiclass classifier** for direct ATSEG classification:

- `SEG_A`
- `SEG_B`
- `SEG_C`

The model uses the same tensor-based feature representation built in previous notebooks. Since XGBoost expects a tabular 2D input, each HCP tensor is flattened from:

`(HCPs, weeks, features)` → `(HCPs, weeks × features)`

The purpose of this notebook is to compare hyperparameter configurations and save the selected final model, metadata, and test predictions.


## 1. Imports

Import the required libraries for tensor loading, data preparation, model training, evaluation, and artifact export.

In [ ]:
import os
import json
import joblib
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import tensorflow as tf
import matplotlib.pyplot as plt

from xgboost import XGBClassifier

from sklearn.model_selection import ParameterSampler
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

## 2. Reproducibility

Set a fixed random seed so that the randomized hyperparameter search and model training are as reproducible as possible.


In [ ]:
SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

## 3. Paths

Define the local folders for the input tensors, trained models, and prediction outputs.

> Update these paths when running the notebook on a different computer.


In [3]:
TENSOR_DIR = Path(r"C:\Users\omarl\Downloads\pfizer_tensors")
MODEL_DIR = Path(r"C:\Users\omarl\Downloads\pfizer_models")
OUTPUT_DIR = Path(r"C:\Users\omarl\Downloads\pfizer_outputs")

MODEL_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 4. Load Tensor Artifacts

Load the precomputed tensors:

- `X_features.pt`: 3D HCP tensor with weekly behavior features
- `y_labels.pt`: encoded ATSEG label (`0=SEG_A`, `1=SEG_B`, `2=SEG_C`, `-1=unlabeled`)
- `folds.pt`: fold assignment used to split by HCP

Using folds avoids leakage across train, validation, and test sets.

In [ ]:
X_torch = torch.load(TENSOR_DIR / "X_features.pt")
y_torch = torch.load(TENSOR_DIR / "y_labels.pt")
fold_torch = torch.load(TENSOR_DIR / "folds.pt")

X_tf = tf.convert_to_tensor(X_torch.cpu().numpy(), dtype=tf.float32)
y_tf = tf.convert_to_tensor(y_torch.cpu().numpy(), dtype=tf.int64)
fold_tf = tf.convert_to_tensor(fold_torch.cpu().numpy(), dtype=tf.int64)

print("X shape:", X_tf.shape)
print("y shape:", y_tf.shape)
print("folds shape:", fold_tf.shape)

C:\Users\omarl\AppData\Local\Temp\ipykernel_21500\3665750501.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  X_torch = torch.load(TENSOR_DIR / "X_features.pt")


X shape: (20931, 86, 65)
y shape: (20931,)
folds shape: (20931,)


C:\Users\omarl\AppData\Local\Temp\ipykernel_21500\3665750501.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  y_torch = torch.load(TENSOR_DIR / "y_labels.pt")
C:\Users\om

## 5. Keep Only Labeled HCPs

For multiclass model training, we only use HCPs with known ATSEG labels. Unlabeled HCPs are excluded from training and evaluation.


In [ ]:
labeled_mask = y_tf != -1

X_labeled = tf.boolean_mask(X_tf, labeled_mask)
y_labeled = tf.boolean_mask(y_tf, labeled_mask)
fold_labeled = tf.boolean_mask(fold_tf, labeled_mask)

target_names = ["SEG_A", "SEG_B", "SEG_C"]

print("X labeled:", X_labeled.shape)
print("y labeled:", y_labeled.shape)
print("fold labeled:", fold_labeled.shape)

print(pd.Series(y_labeled.numpy()).value_counts().sort_index())

X labeled: (11899, 86, 65)
y labeled: (11899,)
fold labeled: (11899,)
0    6406
1    3349
2    2144
Name: count, dtype: int64


## 6. Train / Validation / Test Split by HCP Fold

We use a fixed fold split:

- Train: all folds except validation and test
- Validation: used for hyperparameter selection
- Test: held out until the final evaluation

This keeps model selection separate from final model assessment.

In [ ]:
test_fold = 3
val_fold = 4

train_mask = (fold_labeled != test_fold) & (fold_labeled != val_fold)
val_mask = fold_labeled == val_fold
test_mask = fold_labeled == test_fold

X_train_tensor = tf.boolean_mask(X_labeled, train_mask)
y_train_tensor = tf.boolean_mask(y_labeled, train_mask)

X_val_tensor = tf.boolean_mask(X_labeled, val_mask)
y_val_tensor = tf.boolean_mask(y_labeled, val_mask)

X_test_tensor = tf.boolean_mask(X_labeled, test_mask)
y_test_tensor = tf.boolean_mask(y_labeled, test_mask)

print("Train:", X_train_tensor.shape, y_train_tensor.shape)
print("Val:", X_val_tensor.shape, y_val_tensor.shape)
print("Test:", X_test_tensor.shape, y_test_tensor.shape)

Train: (7140, 86, 65) (7140,)
Val: (2379, 86, 65) (2379,)
Test: (2380, 86, 65) (2380,)


## 7. Flatten Temporal Tensors

XGBoost requires a 2D matrix. Therefore, each HCP's weekly tensor is flattened into a single feature vector.

This keeps the same information but changes the shape from temporal to tabular.

In [ ]:
X_train_flat = X_train_tensor.numpy().reshape(X_train_tensor.shape[0], -1)
X_val_flat = X_val_tensor.numpy().reshape(X_val_tensor.shape[0], -1)
X_test_flat = X_test_tensor.numpy().reshape(X_test_tensor.shape[0], -1)

y_train_mc = y_train_tensor.numpy()
y_val_mc = y_val_tensor.numpy()
y_test_mc = y_test_tensor.numpy()

print("X_train_flat:", X_train_flat.shape)
print("X_val_flat:", X_val_flat.shape)
print("X_test_flat:", X_test_flat.shape)

print("y_train:", pd.Series(y_train_mc).value_counts().sort_index())
print("y_val:", pd.Series(y_val_mc).value_counts().sort_index())
print("y_test:", pd.Series(y_test_mc).value_counts().sort_index())

X_train_flat: (7140, 5590)
X_val_flat: (2379, 5590)
X_test_flat: (2380, 5590)
y_train: 0    3844
1    2009
2    1287
Name: count, dtype: int64
y_val: 0    1281
1     670
2     428
Name: count, dtype: int64
y_test: 0    1281
1     670
2     429
Name: count, dtype: int64


## 8. Evaluation Function

The main metric is **Macro F1**, because it gives equal importance to all three segments. This is important because `SEG_B` and `SEG_C` are harder to separate and should not be ignored by optimizing only accuracy.

The custom score combines:

- Macro F1
- Average recall of `SEG_B` and `SEG_C`
- Weighted F1

This reflects both overall model quality and business relevance for the harder B/C separation.

In [ ]:
def evaluate_multiclass_predictions(y_true, y_pred, y_proba=None):
    report = classification_report(
        y_true,
        y_pred,
        target_names=["SEG_A", "SEG_B", "SEG_C"],
        output_dict=True,
        zero_division=0
    )

    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted"),
        "SEG_A_precision": report["SEG_A"]["precision"],
        "SEG_A_recall": report["SEG_A"]["recall"],
        "SEG_A_f1": report["SEG_A"]["f1-score"],
        "SEG_B_precision": report["SEG_B"]["precision"],
        "SEG_B_recall": report["SEG_B"]["recall"],
        "SEG_B_f1": report["SEG_B"]["f1-score"],
        "SEG_C_precision": report["SEG_C"]["precision"],
        "SEG_C_recall": report["SEG_C"]["recall"],
        "SEG_C_f1": report["SEG_C"]["f1-score"],
    }

    metrics["bc_recall_avg"] = (
        metrics["SEG_B_recall"] + metrics["SEG_C_recall"]
    ) / 2

    metrics["custom_score"] = (
        0.50 * metrics["macro_f1"] +
        0.30 * metrics["bc_recall_avg"] +
        0.20 * metrics["weighted_f1"]
    )

    return metrics

## 9. Hyperparameter Search Space

Define a randomized hyperparameter search for XGBoost.

The search explores number of trees, tree depth, learning rate, row/feature subsampling, regularization, and split constraints.

A randomized search is preferred over a full grid because it explores a wider set of combinations with fewer model trainings.

In [ ]:
xgb_param_grid = {
    "n_estimators": [200, 300, 500, 700],
    "max_depth": [3, 4, 5, 6],
    "learning_rate": [0.01, 0.03, 0.05, 0.07, 0.1],
    "subsample": [0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.6, 0.75, 0.85, 1.0],
    "min_child_weight": [1, 3, 5, 7],
    "gamma": [0, 0.05, 0.1, 0.2],
    "reg_alpha": [0, 0.01, 0.1, 0.5],
    "reg_lambda": [1, 2, 5, 10],
}

n_iter = 100

xgb_param_candidates = list(ParameterSampler(
    xgb_param_grid,
    n_iter=n_iter,
    random_state=SEED
))

len(xgb_param_candidates)

100

## 10. Class Weights

Class weights are used to reduce the impact of class imbalance. They assign higher training weight to underrepresented classes.


In [10]:
sample_weights_mc = compute_sample_weight(
    class_weight="balanced",
    y=y_train_mc
)

## 11. Randomized Search on Validation Fold

Each candidate is trained on the training folds and evaluated on the validation fold. The results are stored in a dataframe for later comparison.

The validation fold is used only for model selection; the test fold remains untouched until the final evaluation.

In [ ]:
xgb_results = []
sample_weights_mc = compute_sample_weight(
    class_weight="balanced",
    y=y_train_mc
)

for i, params in enumerate(xgb_param_candidates, start=1):
    print("\n" + "=" * 80)
    print(f"Training XGB candidate {i}/{len(xgb_param_candidates)}")
    print(params)
    print("=" * 80)

    model = XGBClassifier(
        objective="multi:softprob",
        num_class=3,
        eval_metric="mlogloss",
        tree_method="hist",
        random_state=SEED,
        n_jobs=-1,
        **params
    )

    model.fit(
        X_train_flat,
        y_train_mc,
        sample_weight=sample_weights_mc
    )

    val_proba = model.predict_proba(X_val_flat)
    val_pred = np.argmax(val_proba, axis=1)

    val_metrics = evaluate_multiclass_predictions(
        y_true=y_val_mc,
        y_pred=val_pred,
        y_proba=val_proba
    )

    result = {
        "candidate": i,
        **params,
        **val_metrics
    }

    xgb_results.append(result)

    print(
        f"Macro F1: {val_metrics['macro_f1']:.4f} | "
        f"Weighted F1: {val_metrics['weighted_f1']:.4f} | "
        f"SEG_B recall: {val_metrics['SEG_B_recall']:.4f} | "
        f"SEG_C recall: {val_metrics['SEG_C_recall']:.4f} | "
        f"Custom score: {val_metrics['custom_score']:.4f}"
    )

xgb_results_df = pd.DataFrame(xgb_results)

xgb_results_df.sort_values(
    ["custom_score", "macro_f1", "weighted_f1"],
    ascending=False
).head(15)


Training XGB candidate 1/100
{'subsample': 0.9, 'reg_lambda': 2, 'reg_alpha': 0.1, 'n_estimators': 300, 'min_child_weight': 1, 'max_depth': 6, 'learning_rate': 0.1, 'gamma': 0.05, 'colsample_bytree': 0.75}
Macro F1: 0.5503 | Weighted F1: 0.6376 | SEG_B recall: 0.5701 | SEG_C recall: 0.2383 | Custom score: 0.5239

Training XGB candidate 2/100
{'subsample': 1.0, 'reg_lambda': 1, 'reg_alpha': 0.5, 'n_estimators': 500, 'min_child_weight': 3, 'max_depth': 6, 'learning_rate': 0.01, 'gamma': 0.2, 'colsample_bytree': 0.75}
Macro F1: 0.5711 | Weighted F1: 0.6454 | SEG_B recall: 0.5791 | SEG_C recall: 0.3528 | Custom score: 0.5544

Training XGB candidate 3/100
{'subsample': 0.7, 'reg_lambda': 10, 'reg_alpha': 0.01, 'n_estimators': 300, 'min_child_weight': 7, 'max_depth': 3, 'learning_rate': 0.05, 'gamma': 0.1, 'colsample_bytree': 0.75}
Macro F1: 0.5769 | Weighted F1: 0.6496 | SEG_B recall: 0.5627 | SEG_C recall: 0.3925 | Custom score: 0.5616

Training XGB candidate 4/100
{'subsample': 0.9, 'reg

,candidate,subsample,reg_lambda,reg_alpha,n_estimators,min_child_weight,max_depth,learning_rate,gamma,colsample_bytree,...,SEG_A_recall,SEG_A_f1,SEG_B_precision,SEG_B_recall,SEG_B_f1,SEG_C_precision,SEG_C_recall,SEG_C_f1,bc_recall_avg,custom_score
37,38,1.0,10,0.50,200,7,5,0.05,0.05,0.60,...,0.786105,0.783963,0.581081,0.577612,0.579341,0.397647,0.394860,0.396249,0.486236,0.570446
32,33,0.8,10,0.50,500,1,3,0.03,0.20,0.60,...,0.774395,0.778956,0.592063,0.556716,0.573846,0.376812,0.425234,0.399561,0.490975,0.569940
57,58,0.9,1,0.10,200,1,3,0.03,0.05,0.75,...,0.772053,0.778127,0.588723,0.529851,0.557738,0.378641,0.455607,0.413574,0.492729,0.569486
43,44,0.7,5,0.01,300,5,3,0.03,0.05,1.00,...,0.772053,0.777516,0.592652,0.553731,0.572531,0.371429,0.425234,0.396514,0.489482,0.568186
25,26,1.0,5,0.00,700,3,3,0.01,0.05,0.85,...,0.769711,0.777296,0.587276,0.537313,0.561185,0.370588,0.441589,0.402985,0.489451,0.566898
78,79,1.0,10,0.00,700,5,5,0.01,0.05,0.60,...,0.774395,0.778650,0.575301,0.570149,0.572714,0.383929,0.401869,0.392694,0.486009,0.566722
49,50,0.7,2,0.50,200,5,5,0.03,0.20,0.60,...,0.781421,0.782643,0.570783,0.565672,0.568216,0.390411,0.399533,0.394919,0.482602,0.566243
19,20,0.8,2,0.10,200,7,5,0.03,0.00,1.00,...,0.779859,0.779251,0.577844,0.576119,0.576981,0.387850,0.387850,0.387850,0.481985,0.565650
45,46,0.8,10,0.00,200,1,5,0.07,0.20,0.85,...,0.800937,0.789838,0.581602,0.585075,0.583333,0.396907,0.359813,0.377451,0.472444,0.565001
46,47,0.8,1,0.50,700,1,3,0.01,0.10,0.85,...,0.772053,0.778434,0.580128,0.540299,0.559505,0.369697,0.427570,0.396533,0.483934,0.563873


## 12. Save Hyperparameter Search Results

Save all candidate results so the model selection process is traceable and reproducible.

In [ ]:
xgb_results_path = MODEL_DIR / "xgb_multiclass_hyperparameter_search_results.csv"

xgb_results_df.to_csv(xgb_results_path, index=False)

print("Saved:", xgb_results_path)

Saved: C:\Users\omarl\Downloads\pfizer_models\xgb_multiclass_hyperparameter_search_results.csv


In [13]:
# %%
xgb_param_cols = [
    "n_estimators",
    "max_depth",
    "learning_rate",
    "subsample",
    "colsample_bytree",
    "min_child_weight",
    "gamma",
    "reg_alpha",
    "reg_lambda"
]

best_xgb_params = {
    col: best_xgb_row[col]
    for col in xgb_param_cols
}

int_cols = ["n_estimators", "max_depth", "min_child_weight"]

for col in int_cols:
    best_xgb_params[col] = int(best_xgb_params[col])

for col in best_xgb_params:
    if col not in int_cols:
        best_xgb_params[col] = float(best_xgb_params[col])

best_xgb_params

{'n_estimators': 200,
 'max_depth': 5,
 'learning_rate': 0.05,
 'subsample': 1.0,
 'colsample_bytree': 0.6,
 'min_child_weight': 7,
 'gamma': 0.05,
 'reg_alpha': 0.5,
 'reg_lambda': 10.0}

In [11]:
# %%
X_trainval_flat = np.vstack([X_train_flat, X_val_flat])
y_trainval_mc = np.concatenate([y_train_mc, y_val_mc])

sample_weights_trainval_mc = compute_sample_weight(
    class_weight="balanced",
    y=y_trainval_mc
)

# %%
candidate_58_params = {
    "subsample": 0.9,
    "reg_lambda": 1,
    "reg_alpha": 0.1,
    "n_estimators": 200,
    "min_child_weight": 1,
    "max_depth": 3,
    "learning_rate": 0.03,
    "gamma": 0.05,
    "colsample_bytree": 0.75
}

xgb_candidate_58 = XGBClassifier(
    objective="multi:softprob",
    num_class=3,
    eval_metric="mlogloss",
    tree_method="hist",
    random_state=SEED,
    n_jobs=-1,
    **candidate_58_params
)

xgb_candidate_58.fit(
    X_trainval_flat,
    y_trainval_mc,
    sample_weight=sample_weights_trainval_mc
)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.75, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='mlogloss',
              feature_types=None, gamma=0.05, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.03, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=3,
              max_leaves=None, min_child_weight=1, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=200,
              n_jobs=-1, num_class=3, num_parallel_tree=None, ...)

In [12]:
# %%
test_xgb_proba = xgb_candidate_58.predict_proba(X_test_flat)
test_xgb_pred = np.argmax(test_xgb_proba, axis=1)

test_xgb_metrics = evaluate_multiclass_predictions(
    y_true=y_test_mc,
    y_pred=test_xgb_pred,
    y_proba=test_xgb_proba
)

test_xgb_metrics

{'accuracy': 0.6449579831932774,
 'macro_f1': 0.5792501739473731,
 'weighted_f1': 0.6479920763036838,
 'SEG_A_precision': 0.7828525641025641,
 'SEG_A_recall': 0.7626854020296643,
 'SEG_A_f1': 0.7726374060893634,
 'SEG_B_precision': 0.5787878787878787,
 'SEG_B_recall': 0.5701492537313433,
 'SEG_B_f1': 0.5744360902255639,
 'SEG_C_precision': 0.3728813559322034,
 'SEG_C_recall': 0.41025641025641024,
 'SEG_C_f1': 0.390677025527192,
 'bc_recall_avg': 0.49020283199387676,
 'custom_score': 0.5662843518325863}

In [13]:
# %%
target_names = ["SEG_A", "SEG_B", "SEG_C"]

print(classification_report(
    y_test_mc,
    test_xgb_pred,
    target_names=target_names
))

              precision    recall  f1-score   support

       SEG_A       0.78      0.76      0.77      1281
       SEG_B       0.58      0.57      0.57       670
       SEG_C       0.37      0.41      0.39       429

    accuracy                           0.64      2380
   macro avg       0.58      0.58      0.58      2380
weighted avg       0.65      0.64      0.65      2380



In [14]:

MODEL_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

xgb_model_path = MODEL_DIR / "best_xgb_multiclass_atseg_candidate58.joblib"
xgb_metadata_path = MODEL_DIR / "xgb_multiclass_candidate58_metadata.json"
xgb_test_predictions_path = OUTPUT_DIR / "xgb_multiclass_candidate58_test_predictions.csv"

joblib.dump(xgb_candidate_58, xgb_model_path)

metadata_xgb_candidate58 = {
    "model": "XGBClassifier",
    "task": "SEG_A vs SEG_B vs SEG_C",
    "selection_reason": "Candidate 58 selected due to better SEG_C recall while maintaining competitive Macro F1.",
    "best_params": candidate_58_params,
    "decision_rule": "argmax over predicted class probabilities",
    "label_mapping": {
        "0": "SEG_A",
        "1": "SEG_B",
        "2": "SEG_C"
    },
    "test_metrics": test_xgb_metrics
}

with open(xgb_metadata_path, "w") as f:
    json.dump(metadata_xgb_candidate58, f, indent=4)

test_predictions_xgb_df = pd.DataFrame({
    "true_label_encoded": y_test_mc,
    "true_label": [target_names[i] for i in y_test_mc],
    "pred_label_encoded": test_xgb_pred,
    "pred_label": [target_names[i] for i in test_xgb_pred],
    "prob_SEG_A": test_xgb_proba[:, 0],
    "prob_SEG_B": test_xgb_proba[:, 1],
    "prob_SEG_C": test_xgb_proba[:, 2],
})

test_predictions_xgb_df.to_csv(xgb_test_predictions_path, index=False)

print("Saved model:", xgb_model_path)
print("Saved metadata:", xgb_metadata_path)
print("Saved test predictions:", xgb_test_predictions_path)

Saved model: C:\Users\omarl\Downloads\pfizer_models\best_xgb_multiclass_atseg_candidate58.joblib
Saved metadata: C:\Users\omarl\Downloads\pfizer_models\xgb_multiclass_candidate58_metadata.json
Saved test predictions: C:\Users\omarl\Downloads\pfizer_outputs\xgb_multiclass_candidate58_test_predictions.csv


In [15]:
# %%
manifest_path = TENSOR_DIR / "tensor_manifest.csv"

if manifest_path.exists():
    manifest_df = pd.read_csv(manifest_path)

    labeled_mask_np = labeled_mask.numpy()
    test_mask_np = test_mask.numpy()

    manifest_labeled = manifest_df.loc[labeled_mask_np].reset_index(drop=True)
    manifest_test = manifest_labeled.loc[test_mask_np].reset_index(drop=True)

    assert len(manifest_test) == len(test_predictions_xgb_df)

    cols_to_add = ["NUEVO_ID"]

    for col in ["ATSEG_HCP", "HCP_FOLD"]:
        if col in manifest_test.columns and col not in cols_to_add:
            cols_to_add.append(col)

    test_predictions_xgb_with_id_df = pd.concat(
        [
            manifest_test[cols_to_add].reset_index(drop=True),
            test_predictions_xgb_df.reset_index(drop=True)
        ],
        axis=1
    )

    xgb_test_predictions_with_id_path = OUTPUT_DIR / "xgb_multiclass_candidate58_test_predictions_with_hcp_id.csv"

    test_predictions_xgb_with_id_df.to_csv(
        xgb_test_predictions_with_id_path,
        index=False
    )

    print("Saved test predictions with HCP ID:", xgb_test_predictions_with_id_path)
else:
    print("No tensor_manifest.csv found. Test predictions were saved without HCP ID.")

Saved test predictions with HCP ID: C:\Users\omarl\Downloads\pfizer_outputs\xgb_multiclass_candidate58_test_predictions_with_hcp_id.csv
